# MCP + AI Client

Ein minimaler, aber vollständiger Beispiel-Client, der

* mit dem MCP-Server spricht (Resource + Tool),
* Ollama über die OpenAI-kompatible Chat-API nutzt,
* Tool-Calls des Modells nimmt und sie auf MCP (read_resource / call_tool) ausführt.

👉 Wichtig:
* Als erstes muss `mcp-server` gestartet werden. 
* Die Variable `OLLAMA_BASE_URL` zeigt auf AI_BASE_URL

In [ ]:
%run ~/work/env.py

In [ ]:
import asyncio
import json

from mcp.client.sse import sse_client
from mcp import ClientSession

from openai import AsyncOpenAI

# -----------------------------------------------------------
# Konfiguration
# -----------------------------------------------------------

MCP_SSE_URL = "http://localhost:8000/sse"          # FastMCP-Server (separat starten)
OLLAMA_BASE_URL = AI_BASE_URL                      # OpenAI-kompatible API von Ollama
OLLAMA_API_KEY = OPENAI_API_KEY                    # Dummy, Ollama ignoriert das üblicherweise
OLLAMA_MODEL = AI_MODEL                            # Dein Ollama-Modellname

SYSTEM_PROMPT = """
Du bist ein Einkaufsassistent für einen kleinen Webshop.
Du hast Zugriff auf diese Funktionen:

- list_products: Hole die aktuelle Produktliste.
- buy_product: Kaufe ein Produkt in einer bestimmten Menge.

Arbeite so:
1) Nutze zuerst list_products, um dem Benutzer verfügbare Produkte zu zeigen.
2) Wenn der Benutzer etwas kaufen möchte, nutze buy_product mit der passenden id und quantity.
3) Erkläre dem Benutzer danach in normalem Deutsch, was du gemacht hast.
"""

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "list_products",
            "description": "Liste alle verfügbaren Produkte im Webshop auf.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "buy_product",
            "description": "Kaufe ein Produkt in einer bestimmten Menge.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "Produkt-ID aus der Produktliste, z.B. '1' oder '2'."
                    },
                    "quantity": {
                        "type": "integer",
                        "description": "Anzahl der zu kaufenden Stück.",
                        "minimum": 1
                    },
                },
                "required": ["id", "quantity"],
            },
        },
    },
]

llm_client = AsyncOpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key=OLLAMA_API_KEY,
)


# -----------------------------------------------------------
# Helper: MCP-Resultate JSON-fähig machen
# -----------------------------------------------------------

def to_jsonable(obj):
    """Macht MCP-Resultate (Pydantic-Modelle etc.) JSON-serialisierbar."""

    # Pydantic v2: sauber als JSON-kompatible Struktur dumpen
    if hasattr(obj, "model_dump"):
        try:
            dumped = obj.model_dump(mode="json")
        except TypeError:
            dumped = obj.model_dump()
        return to_jsonable(dumped)

    # Pydantic v1: dict()
    if hasattr(obj, "dict"):
        return to_jsonable(obj.dict())

    # Listen rekursiv behandeln
    if isinstance(obj, list):
        return [to_jsonable(x) for x in obj]

    # Dicts rekursiv behandeln
    if isinstance(obj, dict):
        return {k: to_jsonable(v) for k, v in obj.items()}

    # JSON-Grundtypen durchlassen
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj

    # Alles andere (z.B. AnyUrl, Enums, sonstige Klassen) als String darstellen
    return str(obj)

# -----------------------------------------------------------
# LLM-Aufruf über OpenAI-Library
# -----------------------------------------------------------

async def call_llm(messages, tools=None):
    response = await llm_client.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto",
    )
    return response.choices[0].message


# -----------------------------------------------------------
# Agent-Loop: User <-> LLM <-> MCP
# -----------------------------------------------------------

async def run_agent():
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ]

    async with sse_client(MCP_SSE_URL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            print("Webshop-Agent gestartet. Tippe 'exit' zum Beenden.\n")

            while True:
                user_input = input("Du: ").strip()
                if user_input.lower() in ("exit", "quit"):
                    print("Beende Agent.")
                    break
                if not user_input:
                    continue

                messages.append({"role": "user", "content": user_input})

                # 1. Call: Modell entscheidet, ob Tools verwendet werden
                try:
                    assistant_msg = await call_llm(messages, tools=TOOLS)
                except Exception as e:
                    print(f"[Fehler] Problem beim Aufruf des Modells: {e}")
                    continue

                tool_calls = assistant_msg.tool_calls or []

                # Fall A: keine Tool-Calls -> direkte Antwort
                if not tool_calls:
                    content = assistant_msg.content or ""
                    print("Assistent:", content.strip())
                    messages.append(
                        {"role": "assistant", "content": content}
                    )
                    continue

                # Fall B: es gibt Tool-Calls -> via MCP ausführen
                messages.append({
                    "role": "assistant",
                    "content": assistant_msg.content or "",
                    "tool_calls": [
                        {
                            "id": tc.id,
                            "type": tc.type,
                            "function": {
                                "name": tc.function.name,
                                "arguments": tc.function.arguments,
                            },
                        }
                        for tc in tool_calls
                    ],
                })

                # Tool-Calls nacheinander bearbeiten
                for tc in tool_calls:
                    tool_name = tc.function.name
                    raw_args = tc.function.arguments or "{}"

                    try:
                        args = json.loads(raw_args)
                    except json.JSONDecodeError:
                        print(f"[Warnung] Konnte Tool-Argumente nicht parsen: {raw_args}")
                        args = {}

                    print(f"[Debug] Tool-Call vom Modell: {tool_name}({args})")

                    # Mapping Tool-Name -> MCP-Aufruf
                    if tool_name == "list_products":
                        # ReadResourceResult -> jsonable
                        result = await session.read_resource("products://")
                    else:
                        # CallToolResult (oder Liste) -> jsonable
                        result = await session.call_tool(tool_name, args)

                    result_for_llm = to_jsonable(result)

                    messages.append({
                        "role": "tool",
                        "tool_call_id": tc.id,
                        "name": tool_name,
                        "content": json.dumps(result_for_llm, ensure_ascii=False),
                    })

                # 2. Call: Modell bekommt Tool-Ergebnisse und soll normale Antwort formulieren
                try:
                    followup_msg = await call_llm(messages, tools=TOOLS)
                except Exception as e:
                    print(f"[Fehler] Problem beim zweiten Modellaufruf: {e}")
                    continue

                final_content = followup_msg.content or ""
                print("Assistent:", final_content.strip())
                messages.append(
                    {"role": "assistant", "content": final_content}
                )


# In Jupyter ausführen:
await run_agent()
